# INNOBEST Huánuco 2026 — MobileNetV3 + YOLO11-cls

**Objetivo:** ejecutar **100 épocas por modelo** sobre las **mismas particiones de datos** (70 % entrenamiento, 15 % validación, 15 % prueba aproximadamente). **No pulses «Ejecutar todas» antes de verificar las fotografías.**

Las fotos originales permanecen intactas en Google Drive. Esta edición del proyecto es independiente de la versión antigua. Dos redes clasifican una fotografía completa: YOLO-cls **no dibuja recuadros**.


### 1. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Comprobar GPU y fotografías subidas al Drive

In [ ]:
from pathlib import Path
import torch

DRIVE = Path('/content/drive/MyDrive')
DATOS = DRIVE / 'innobest_huanuco_2026' / 'dataset'  # Carpeta que ya usa tu proyecto
CLASES = ['ara_militaris', 'pauxi_koepckeae', 'spizaetus_isidori']
EXT = {'.jpg', '.jpeg', '.png', '.webp'}

print('GPU CUDA:', torch.cuda.is_available())
print('Dispositivo:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'SIN GPU')
print('Carpeta de imágenes:', DATOS)
if not DATOS.is_dir():
    raise FileNotFoundError('No existe la carpeta dataset. Sube tus carpetas de aves a ese directorio de Drive.')
for clase in CLASES:
    carpeta = DATOS / clase
    fotos = sorted(p for p in carpeta.iterdir() if p.is_file() and p.suffix.lower() in EXT) if carpeta.is_dir() else []
    print(f'{clase}: {len(fotos)} imágenes')
    if not fotos:
        raise ValueError(f'Falta subir fotografías en {carpeta}. No avances todavía.')
if not torch.cuda.is_available():
    raise RuntimeError('Activa NVIDIA T4 antes del entrenamiento.')
print('LISTO: carpetas encontradas. La unicidad y validez se verifican en el paso 5.')

### 3. Instalar versión actualizada del proyecto sin sobrescribir la carpeta antigua
Sube `innobest_huanuco_2026_dos_modelos.zip` a **Mi unidad** antes de ejecutar.

In [ ]:
from pathlib import Path
import zipfile
ZIP = DRIVE / 'innobest_huanuco_2026_dos_modelos.zip'
PROYECTO = DRIVE / 'innobest_huanuco_2026_dos_modelos'
if not PROYECTO.is_dir():
    if not ZIP.is_file():
        raise FileNotFoundError(f'Primero sube el ZIP a Google Drive: {ZIP.name}')
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(DRIVE)  # ZIP creado para este proyecto; no usar ZIPs ajenos no confiables
assert (PROYECTO / 'entrenar_yolo.py').is_file(), 'No está entrenar_yolo.py'
assert (PROYECTO / 'preparar_datos.py').is_file(), 'No está preparar_datos.py'
print('Proyecto actualizado disponible:', PROYECTO)

### 4. Instalar dependencias de los dos modelos

In [ ]:
%cd /content/drive/MyDrive/innobest_huanuco_2026_dos_modelos
!pip -q install -r requirements.txt
import ultralytics
print('Ultralytics:', ultralytics.__version__)

### 5. Crear una sola división de imágenes para ambos modelos
Las imágenes originales **no se borran**. Retira copias exactas del experimento y agrupa fotografías extremadamente parecidas en una misma partición. Los archivos del dataset preparado se almacenan temporalmente en `/content` para mejorar la velocidad. Guarda inventario y descarte en Drive.

In [ ]:
import subprocess, sys
PREPARADO = Path('/content/innobest_preparado')
REPORTE = PROYECTO / 'modelos' / 'auditoria_dataset'
subprocess.run([sys.executable, str(PROYECTO/'preparar_datos.py'),
                '--datos', str(DATOS), '--destino', str(PREPARADO),
                '--reporte', str(REPORTE), '--recrear'], check=True)
print('Listo. Revisa auditoria_dataset/resumen_datos.json en tu Drive.')

### 6. Entrenar MobileNetV3 durante 100 épocas (80 + 20)
T4 habilitada. Los resultados y el mejor modelo se guardan en Drive. No entrenes si sospechas que hay etiquetas de especies equivocadas.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, str(PROYECTO/'entrenar.py'),
                '--datos', str(PREPARADO), '--salida', str(PROYECTO/'modelos'/'mobilenet'),
                '--epocas-cabecera', '80', '--epocas-ajuste', '20', '--batch', '16'], check=True)

### 7. Entrenar YOLO11n-cls durante 100 épocas
Para comparar limpiamente, se usan **exactamente las mismas imágenes de prueba** de MobileNetV3. YOLO guarda sus checkpoints cada 10 épocas.

In [ ]:
subprocess.run([sys.executable, str(PROYECTO/'entrenar_yolo.py'),
                '--datos', str(PREPARADO), '--salida', str(PROYECTO/'modelos'/'yolo_cls'),
                '--pesos', 'yolo11n-cls.pt'],check=True)

### 8. Comparar métricas reales en conjunto de prueba separado

In [ ]:
import json
import pandas as pd
salidas = {'MobileNetV3': PROYECTO/'modelos'/'mobilenet'/'metricas.json',
           'YOLO11n-cls': PROYECTO/'modelos'/'yolo_cls'/'metricas.json'}
filas = []
for nombre, ruta in salidas.items():
    if not ruta.exists():
        print('Aún sin resultados:', nombre)
        continue
    m = json.loads(ruta.read_text(encoding='utf-8'))
    filas.append({'Modelo': nombre, 'Épocas': m['epocas_ejecutadas'],
                  'Accuracy prueba': m['accuracy_test'],
                  'F1 macro prueba': m['f1_macro_test'],
                  'Muestras prueba': m['n_prueba']})
display(pd.DataFrame(filas))
print('Los resultados no representan censos de aves ni avistamientos confirmados.')